# Home Advantage in European Football
**Programming for Data Science · Final Project**

**Team** — Sam Olivares, Shayan Karimi, Etienne Padelo, Kerry Assohoun

**Research question.** How does home advantage vary between Ligue 1 and the Premier League, across weeks and between groups of nearby stadiums in 2024–2025?

The dataset combines Football-Data.co.uk match results with the coordinates of each club's usual home stadium. Each row represents a match. The dataset covers 686 matches and 38 stadiums. Venue changes for individual matches are outside the scope of the data.

The KDD workflow covers data selection, validation, transformation, analysis and visualization.

In [ ]:
from collections import Counter
from html import escape
from io import StringIO
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

DATASET = Path("Football_PDS_2024-2025_with_stadiums.csv")
TEAM_MEMBERS = ["Sam Olivares", "Shayan Karimi", "Etienne Padelo", "Kerry Assohoun"]
COLORS = ["#196F68", "#E6AD55", "#536E9D"]
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})

## Data Loading and Exploration

`FTR` is the full-time result (H for home win, D for draw, A for away win). `FTHG` and `FTAG` are full-time goals, `HS` and `AS` are total shots, and `HST` and `AST` are shots on target. Each pair gives the home value first. Stadium coordinates support the spatial analysis. The `coordinate_source` column provides a source URL for each location.

In [ ]:
def load_matches(path):
    """Load the CSV at path and return validated matches with parsed dates and signed outcomes."""
    data = pd.read_csv(path, dtype={"season": str})
    data["Date"] = pd.to_datetime(data["Date"], dayfirst=True, errors="raise")
    columns = ["league", "season", "Date", "HomeTeam", "AwayTeam"]
    if data.duplicated(columns).any():
        raise ValueError("Duplicate matches found.")
    if not data["FTR"].isin(["H", "D", "A"]).all():
        raise ValueError("Missing or unrecognized match outcome.")
    return data.assign(result_value=data["FTR"].map({"H": 1, "D": 0, "A": -1}))


def describe_matches(data):
    """Return data types, missing-value counts and ranges for the source columns in data."""
    rows = []
    for name, values in data.drop(columns="result_value").items():
        valid = values.dropna()
        if pd.api.types.is_numeric_dtype(values) or pd.api.types.is_datetime64_any_dtype(values):
            extent = f"{valid.min()} → {valid.max()}"
        else:
            levels = sorted(valid.astype(str).unique())
            extent = ", ".join(levels) if len(levels) <= 5 else f"{len(levels)} categories"
        rows.append((name, str(values.dtype), int(values.isna().sum()), extent))
    return pd.DataFrame(rows, columns=["Column", "Type", "Missing", "Range or categories"])

## League Comparison

Home advantage is the home win rate minus the away win rate, in percentage points. A match is assigned +1 for a home win, 0 for a draw and −1 for an away win. The mean of these values multiplied by 100 equals the indicator. Draws are included in the denominator.

In [ ]:
def outcome_summary(data, by):
    """Group matches by the supplied columns and return counts, rates and the home–away gap."""
    counts = pd.crosstab([data[key] for key in by], data["FTR"]).reindex(columns=["H", "D", "A"], fill_value=0)
    rates = counts.div(counts.sum(axis=1), axis=0).mul(100)
    return rates.assign(Matches=counts.sum(axis=1), Gap=rates["H"] - rates["A"])


def league_chart(summary):
    """Plot the three match outcome rates from the league summary."""
    ax = summary[["H", "D", "A"]].rename(columns={"H": "Home win", "D": "Draw", "A": "Away win"}).plot.bar(
        figsize=(7.3, 3.8), color=COLORS, rot=0)
    ax.set(xlabel="", ylabel="Share of matches (%)", title="Match Outcomes by League")
    ax.legend(frameon=False, ncol=3)
    return ax.figure

## Frequent Pattern Mining

A match becomes a transaction containing its league, the side with more shots, the side with more shots on target and the final result. We enumerate two- and three-item combinations and retain rules with a home win as the consequent. Support is the share of matches with both the antecedent and a home win. Confidence is the share of home wins among matches with the antecedent. Lift divides confidence by the overall home win rate. We use a minimum support of 8%. Shots are measured during the match, so the rules describe associations.

In [ ]:
def frequent_rules(data, min_support=0.08):
    """Find rules associated with home wins and return their support, confidence and lift."""
    labels = {1: "home", 0: "equal", -1: "away"}
    shots = np.sign(data["HS"] - data["AS"]).map(labels)
    on_target = np.sign(data["HST"] - data["AST"]).map(labels)
    total, wins = Counter(), Counter()
    for league, shot, target, result in zip(data["league"], shots, on_target, data["FTR"]):
        items = (league, f"Shots: {shot}", f"Shots on target: {target}")
        for size in (1, 2):
            for antecedent in combinations(items, size):
                total[antecedent] += 1
                wins[antecedent] += int(result == "H")
    base_rate = data["FTR"].eq("H").mean()
    rows = []
    for antecedent, count in total.items():
        support = wins[antecedent] / len(data)
        if support >= min_support:
            confidence = wins[antecedent] / count
            rows.append((" + ".join(antecedent), count, support, confidence, confidence / base_rate))
    return pd.DataFrame(rows, columns=["Pattern", "Matches", "Support", "Confidence", "Lift"]).sort_values(
        ["Lift", "Support"], ascending=False, ignore_index=True)


def rule_chart(rules):
    """Plot lift for the five highest-ranked association rules."""
    top = rules.head(5).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.3, 3.8), layout="constrained")
    ax.barh(top["Pattern"], top["Lift"], color=COLORS[0])
    ax.axvline(1, color="#555555", linestyle="--")
    ax.set(xlabel="Lift relative to the overall home win rate", title="Patterns Associated with Home Wins")
    return fig

## Weekly Forecasting

We pool both leagues by Monday–Sunday week and keep weeks containing at least one match. For each target week, the forecast uses the average signed match result over the preceding four active weeks. The baseline uses all earlier matches. We evaluate both methods over the final eight active weeks with mean absolute error (MAE), in percentage points. This is exploratory because the data cover one season and match counts vary across weeks.

In [ ]:
def weekly_forecast(data, window=4, holdout=8):
    """Return the weekly series, the holdout period and the MAE of both forecasts."""
    date = data["Date"]
    start = (date - pd.to_timedelta(date.dt.weekday, unit="D")).rename("Week")
    weekly = data.groupby(start)["result_value"].agg(Total="sum", Matches="size")
    past_sum = weekly["Total"].shift()
    past_count = weekly["Matches"].shift()
    weekly["Observed"] = 100 * weekly["Total"] / weekly["Matches"]
    weekly["Forecast"] = 100 * past_sum.rolling(window).sum() / past_count.rolling(window).sum()
    weekly["Baseline"] = 100 * past_sum.cumsum() / past_count.cumsum()
    test = weekly.tail(holdout)
    mae = test[["Forecast", "Baseline"]].sub(test["Observed"], axis=0).abs().mean()
    return weekly, test, mae


def forecast_chart(weekly, test):
    """Plot weekly observations and predictions for the holdout period."""
    fig, ax = plt.subplots(figsize=(7.3, 3.8), layout="constrained")
    ax.plot(weekly.index, weekly["Observed"], color=COLORS[0], label="Observed")
    ax.plot(test.index, test["Forecast"], color=COLORS[1], marker="o", label="Forecast")
    ax.plot(test.index, test["Baseline"], color=COLORS[2], linestyle="--", label="Baseline")
    ax.axvspan(test.index[0], test.index[-1], alpha=0.09, color=COLORS[2])
    ax.set(ylabel="Gap (percentage points)", title="Weekly Home Advantage and Holdout Period")
    ax.legend(frameon=False, ncol=3)
    return fig

## Spatial Clustering

DBSCAN clusters stadiums using Haversine distances. The neighborhood radius is 85 km and the minimum is two stadiums, including the stadium itself. Stadiums assigned to noise are shown as unclustered. Match outcomes are then summarized by the host stadium's cluster. Differences can reflect the teams and leagues represented in each cluster.

In [ ]:
def spatial_groups(data, radius_km=85):
    """Cluster the stadiums in data and return their locations and grouped outcomes."""
    places = data[["stadium", "latitude", "longitude"]].drop_duplicates().sort_values("stadium").copy()
    coordinates = np.radians(places[["latitude", "longitude"]].to_numpy())
    labels = DBSCAN(eps=radius_km / 6371.0, min_samples=2, metric="haversine").fit_predict(coordinates)
    places["Cluster"] = [f"Cluster {label + 1}" if label >= 0 else "Unclustered" for label in labels]
    joined = data.merge(places, on=["stadium", "latitude", "longitude"], validate="many_to_one")
    summary = outcome_summary(joined, ["Cluster"]).join(places.groupby("Cluster").size().rename("Stadiums"))
    return places, summary


def spatial_chart(places):
    """Plot stadium locations and their cluster assignments by longitude and latitude."""
    fig, ax = plt.subplots(figsize=(7.3, 4.4), layout="constrained")
    for name, group in places.groupby("Cluster"):
        style = {"color": "#9b9b9b", "marker": "x"} if name == "Unclustered" else {}
        ax.scatter(group["longitude"], group["latitude"], s=36, label=name, **style)
    ax.set(xlabel="Longitude (°)", ylabel="Latitude (°)", title="Geographic Distribution of Stadiums")
    ax.set_aspect(1 / np.cos(np.radians(49)))
    ax.legend(frameon=False, fontsize=8, ncol=2)
    return fig

## Dashboard and Execution

`main()` runs the four analyses, displays the figures and exports the standalone dashboard. The HTML file opens in a browser without a server. Results describe the observed matches and the raw indicator can reflect differences in team strength.

In [ ]:
def export_dashboard(panels, output):
    """Write a standalone HTML dashboard with the figures and tables from panels."""
    nav, sections = [], []
    for index, (title, note, figure, table) in enumerate(panels, start=1):
        target = f"analysis-{index}"
        image = StringIO()
        figure.savefig(image, format="svg", bbox_inches="tight")
        svg = image.getvalue()[image.getvalue().index("<svg"):]
        table_html = table.to_html(float_format=lambda value: f"{value:.2f}", border=0)
        nav.append(f'<a href="#{target}">{escape(title)}</a>')
        sections.append(f'<section id="{target}"><h2>{escape(title)}</h2><p>{escape(note)}</p>'
                        f'<img alt="{escape(title)}" src="data:image/svg+xml;base64,{svg_base64(svg)}">'
                        f'<div class="table">{table_html}</div></section>')
    team = escape(", ".join(TEAM_MEMBERS))
    style = """
    body{font:16px system-ui;color:#182626;background:#f2f6f4;margin:0}
    header{background:#143c39;color:white;padding:3rem max(5%,calc((100% - 1040px)/2))}
    h1{margin:0 0 1rem;font-size:clamp(2rem,5vw,3.3rem)} main{max-width:1040px;margin:auto;padding:1.5rem}
    nav{display:flex;gap:1rem;flex-wrap:wrap} a{color:#196f68} section{background:white;margin:1.3rem 0;padding:1.4rem;border-radius:12px}
    section img{display:block;width:100%;max-width:740px;margin:auto} .table{overflow:auto}
    table{border-collapse:collapse;width:100%;font-size:.9rem} td,th{padding:.45rem;text-align:left;border-bottom:1px solid #e3e8e5}
    """
    output.write_text(f'<!doctype html><html lang="en"><meta charset="utf-8">'
                      f'<meta name="viewport" content="width=device-width,initial-scale=1">'
                      f'<title>Home Advantage</title><style>{style}</style>'
                      f'<header><h1>Home Advantage</h1><p>Ligue 1 and Premier League · Season 2024–2025</p>'
                      f'<p>{team} · {escape(DATASET.name)} · Football-Data.co.uk and stadium coordinates</p></header>'
                      f'<main><nav>{"".join(nav)}</nav>{"".join(sections)}</main></html>', encoding="utf-8")


def svg_base64(svg):
    """Encode SVG text for embedding in a standalone HTML image."""
    from base64 import b64encode
    return b64encode(svg.encode("utf-8")).decode("ascii")


def main(show=True):
    """Run the analyses, export the dashboard and return the data and results."""
    data = load_matches(DATASET)
    profile = describe_matches(data)
    league = outcome_summary(data, ["league"])
    rules = frequent_rules(data)
    weekly, test, mae = weekly_forecast(data)
    places, spatial = spatial_groups(data)
    rule_display = rules.head(8).copy()
    rule_display[["Support", "Confidence"]] *= 100
    rule_display = rule_display.rename(columns={"Support": "Support (%)", "Confidence": "Confidence (%)"})
    panels = [
        ("League Comparison", "Home win, draw and away win rates use all matches. The gap is the home win rate minus the away win rate.", league_chart(league), league),
        ("Frequent Patterns", "A lift above 1 means home wins are more common when the pattern is present.", rule_chart(rules), rule_display),
        ("Forecast", "Mean absolute error of both methods over the final eight active weeks.", forecast_chart(weekly, test), mae.to_frame("MAE (points)")),
        ("Stadiums", "The neighborhood radius is 85 km. Unclustered stadiums were assigned to DBSCAN noise.", spatial_chart(places), spatial),
    ]
    export_dashboard(panels, Path("Olivares_dashboard.html"))
    if show:
        from IPython.display import display
        print(f"{len(data)} matches and {len(profile)} source columns")
        display(profile, league.round(2), rule_display.round(2), mae.round(2), spatial.round(2))
        plt.show()
    return data, league, rules, weekly, test, mae, places, spatial, panels


results = main()

## Conclusions

In 2024–2025, the home–away win gap was +13.7 percentage points in Ligue 1 and +6.1 in the Premier League. In Ligue 1, the home team won 67.9% of matches in which it had more shots on target than its opponent.

The four-week forecast had a mean absolute error of 19.54 percentage points, compared with 20.09 for the baseline. The difference was small over eight test weeks. DBSCAN identified eight clusters and left twelve stadiums unclustered. Results for clusters with two clubs should be interpreted with their sample sizes and club membership in mind.